# Make pseudobulk / cluster bigwig files

**(2026-01-29):** you can generate one BigWig per pre-split scATAC fragments TSV in
`~/Work/projects/Landscapes/scCandT/scATAC_fragments/atac_blk`.

For that workflow, run:
1) the import cell,
2) the `chrom_sizes` cell,
3) the helper-functions cell (`create_bigwig`),
4) the section **"Create BigWig files from pre-split scATAC fragment TSVs"**.



In [1]:
import os
from pathlib import Path
import gzip
import shutil

import pandas as pd
import pyBigWig
import pybedtools

# Optional (only needed for the earlier pseudobulk-from-h5ad part of this notebook)
# import anndata as ad
# import numpy as np
# import scanpy as scan


In [4]:
chrom_sizes = [("chr2L", 23513712),
               ("chr2R", 25286936),
               ("chr3L", 28110227),
               ("chr3R", 32079331),
               ("chr4", 1348131),
               ("chrX", 23542271),
              ("chrY", 3667352)]  

In [27]:
!echo -e "chr2L\t23513712\nchr2R\t25286936\nchr3L\t28110227\nchr3R\t32079331\nchrX\t23542271" > ../files_etc/dm6.chrom.sizes


In [5]:
def read_fragment_file(bed_file):
    """Read a fragments file into a pandas DataFrame."""
    cols = ["chr", "start", "end", "read"]
    df = pd.read_csv(bed_file, sep="\t", header=None, usecols=[0, 1, 2, 3], names=cols, comment="#")
    df["chr"] = df["chr"].astype(str).apply(lambda x: x if x.startswith("chr") else f"chr{x}")
    return df

def filter_reads(bed_df, read_names_file):
    """Filter reads based on a read names file."""
    with open(read_names_file, "r") as f:
        valid_reads = set(line.strip() for line in f)

    # Assuming the BED file has a read name in column 3 (adjust if necessary)
    return bed_df[bed_df["read"].astype(str).isin(valid_reads)]

def create_bigwig(filtered_bed_path, windows_path, output_bw, chrom_sizes, norm = 1):
    """Convert the filtered BED data to a BigWig coverage track."""
    bedf = pybedtools.bedtool.BedTool(filtered_bed_path)
    bedsrt = bedf.sort()
    windf = pybedtools.bedtool.BedTool(windows_path)
    cov = windf.coverage(bedsrt)
    df = cov.to_dataframe(names=['chrom', 'start', 'end', 'depth', 'n', 'chromsize', 'fraction'])
    df = df[(df['depth'] > 0) & (df['chrom'].isin([i[0] for i in chrom_sizes]))]
    
    bw = pyBigWig.open(output_bw, "w")

    # Define chromosome sizes (adjust accordingly)
    bw.addHeader(chrom_sizes)

    bw.addEntries(
        df['chrom'].tolist(),
        starts = (df['start'] + 1).tolist(),
        ends = df['end'].tolist(),
        values = (df['depth']/norm).astype('float').tolist()
    )

    bw.close()


## Create BigWig files from pre-split scATAC fragment TSVs

This section takes a folder with one fragments `.tsv` per cluster and writes one `.bw` per file.
It assumes the TSV is BED-like (at least 3 columns: chrom, start, end; extra columns are ignored).

Edit `input_dir` below to point to your folder:
`~/Work/projects/Landscapes/scCandT/scATAC_fragments/atac_blk`


In [6]:
from pathlib import Path
import gzip
import tempfile
import shutil

# Folder with 5 cluster TSVs (fragments)
input_dir = Path("~/Work/projects/Landscapes/scCandT/scATAC_fragments/atac_blk").expanduser()
# Where to write bigWigs
output_dir = Path("../bw_atac_blk")
output_dir.mkdir(parents=True, exist_ok=True)

# Bin size in bp for the coverage track
bin_size = 50

def write_windows_bed(chrom_sizes, bin_size, out_bed):
    """Create fixed-width genome windows BED from chrom_sizes."""
    out_bed = Path(out_bed)
    out_bed.parent.mkdir(parents=True, exist_ok=True)
    with out_bed.open("w") as f:
        for chrom, size in chrom_sizes:
            for start in range(0, int(size), int(bin_size)):
                end = min(start + int(bin_size), int(size))
                f.write(f"{chrom}\t{start}\t{end}\n")
    return str(out_bed)

windows_bed = Path(f"../bed/dm6.windows_{bin_size}bp.bed")
if not windows_bed.exists():
    _ = write_windows_bed(chrom_sizes, bin_size, windows_bed)

frag_files = sorted(
    list(input_dir.glob("*.tsv")) +
    list(input_dir.glob("*.bed")) +
    list(input_dir.glob("*.tsv.gz")) +
    list(input_dir.glob("*.bed.gz"))
)

if len(frag_files) == 0:
    raise FileNotFoundError(f"No .tsv/.bed(.gz) files found in: {input_dir}")

def count_fragments(path: Path) -> int:
    """Count non-empty, non-comment lines."""
    opener = gzip.open if path.suffix == ".gz" else open
    n = 0
    with opener(path, "rt") as fh:
        for line in fh:
            if not line.strip():
                continue
            if line.startswith("#") or line.startswith("track") or line.startswith("browser"):
                continue
            n += 1
    return n

def maybe_unzip(path: Path) -> Path:
    """pybedtools/bedtools typically expects plain text; unzip .gz to a temp file if needed."""
    if path.suffix != ".gz":
        return path
    tmp_dir = output_dir / "_tmp_unzipped"
    tmp_dir.mkdir(exist_ok=True)
    out_path = tmp_dir / path.name.replace(".gz", "")
    if not out_path.exists():
        with gzip.open(path, "rb") as f_in, open(out_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
    return out_path

for frag_path in frag_files:
    name = frag_path.name
    stem = name.replace(".tsv.gz", "").replace(".bed.gz", "").replace(".tsv", "").replace(".bed", "")
    out_bw = output_dir / f"{stem}.bw"

    # CPM normalization by total fragments in that file
    n_frags = count_fragments(frag_path)
    norm = max(n_frags / 1_000_000, 1e-9)

    frag_plain = maybe_unzip(frag_path)

    print(f"[{stem}] fragments={n_frags:,} -> {out_bw}")
    create_bigwig(str(frag_plain), str(windows_bed), str(out_bw), chrom_sizes, norm=norm)

print("Done.")


[Epidermis] fragments=33,281,509 -> ../bw_atac_blk/Epidermis.bw
[Fat Body] fragments=3,691,710 -> ../bw_atac_blk/Fat Body.bw
[Midgut] fragments=17,283,466 -> ../bw_atac_blk/Midgut.bw
[Muscle] fragments=22,754,142 -> ../bw_atac_blk/Muscle.bw
[Ventral Nerve Cord] fragments=22,997,134 -> ../bw_atac_blk/Ventral Nerve Cord.bw
Done.
